## Simpsons MNIST Perceptron Classification

---

In [ ]:
import os
import random
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
# import seaborn as sns

RANDOM_SEED: int = 42
TEST_SIZE: float = 0.2

EPOCHS: int = 50

LEARNING_RATE: float = 0.01
LEARNING_RATE_SMALL_STEP: float = LEARNING_RATE
LEARNING_RATE_BIG_STEP: float = 0.1
LEARNING_RATES: list[float] = [LEARNING_RATE_SMALL_STEP, LEARNING_RATE_BIG_STEP]

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

LABEL_COUNT: int = 10

# Mapping from label index to character name
label_map: dict[int, str] = {
    0: "bart_simpson",
    1: "charles_montgomery_burns",
    2: "homer_simpson",
    3: "krusty_the_clown",
    4: "lisa_simpson",
    5: "marge_simpson",
    6: "milhouse_van_houten",
    7: "moe_szyslak",
    8: "ned_flanders",
    9: "principal_skinner"
}

### 1. Data Processing

In [ ]:
# Reverse map: name -> index
label_map_inv = {v: k for k, v in label_map.items()}

def load_dataset(base_path, mode = "grayscale"):
    X, y = [], []
    train_path = os.path.join(base_path, "train")

    for label_name in os.listdir(train_path):
        label_dir = os.path.join(train_path, label_name)
        if not os.path.isdir(label_dir):
            continue
        label = label_map_inv[label_name]

        for file in os.listdir(label_dir):
            img_path = os.path.join(label_dir, file)
            img = Image.open(img_path)
            if mode == "grayscale":
                img = img.convert("L")
            else:
                img = img.convert("RGB")

            arr = np.array(img).flatten().astype(np.float32)
            X.append(arr)
            y.append(label)

    X = np.array(X)
    y = np.array(y)

    return X, y

# Usage
X_gray, y_gray = load_dataset("data/grayscale", mode = "grayscale")
X_rgb, y_rgb = load_dataset("data/rgb", mode = "rgb")

# Split into train and validation
X_train_gray, X_val_gray, y_train_gray, y_val_gray = train_test_split(X_gray, y_gray, test_size = TEST_SIZE, random_state = RANDOM_SEED)
X_train_rgb, X_val_rgb, y_train_rgb, y_val_rgb = train_test_split(X_rgb, y_rgb, test_size = TEST_SIZE, random_state = RANDOM_SEED)

### 2. Perceptron Implementation

In [ ]:
class BinaryPerceptron:
    def __init__(self, n_features, learning_rate = LEARNING_RATE):
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.lr = learning_rate

    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return np.where(linear_output >= 0, 1, 0)

    def update(self, X, y):
        for xi, target in zip(X, y):
            pred = self.predict(xi)
            update = self.lr * (target - pred)
            self.weights += update * xi
            self.bias += update

class MultiClassPerceptron:
    def __init__(self, n_classes, n_features, learning_rate = LEARNING_RATE):
        self.classes = n_classes
        self.models = [BinaryPerceptron(n_features, learning_rate) for _ in range(n_classes)]

    def predict(self, X):
        scores = [np.dot(X, model.weights) + model.bias for model in self.models]
        return np.argmax(scores, axis=0)

    def update(self, X, y):
        for xi, target in zip(X, y):
            for i, model in enumerate(self.models):
                binary_label = 1 if target == i else 0
                pred = model.predict(xi)
                update = model.lr * (binary_label - pred)
                model.weights += update * xi
                model.bias += update

### 3. Training Loop

In [ ]:
def train_perceptron(model, X, y, X_train, y_train, epochs = EPOCHS, stop_criteria = "epochs", patience = 3):
    """
    Train perceptron with support for:
    - Fixed number of epochs (stop_criteria="epochs")
    - Early stopping based on validation accuracy (stop_criteria="early_stop")

    patience: number of epochs with no improvement before stopping early.
    """
    history = {"train_acc": [], "val_acc": []}
    best_val = 0
    patience_counter = 0

    for epoch in range(epochs):
        # Shuffle training data
        indices = np.arange(len(X))
        np.random.shuffle(indices)
        X, y = X[indices], y[indices]

        # Perceptron update
        model.update(X, y)

        # Compute accuracies
        train_acc = np.mean([model.predict(xi) == yi for xi, yi in zip(X, y)])
        val_acc = np.mean([model.predict(xi) == yi for xi, yi in zip(X_train, y_train)])
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1}/{epochs}, Train Acc: {train_acc:.3f}, Val Acc: {val_acc:.3f}")

        # --- Early stopping ---
        if stop_criteria == "early_stop":
            if val_acc > best_val:
                best_val = val_acc
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    break

    return history

### 4. Hyperparameter Tuning (Grid Search Skeleton)

In [ ]:
def normalise_data(X, method = "none"):
    """
    Apply different normalisation strategies.
    method: "none", "minmax", or "zscore"
    """
    if method == "none":
        return X
    elif method == "minmax":
        X_min = X.min(axis = 1, keepdims = True)
        X_max = X.max(axis = 1, keepdims = True)
        
        return (X - X_min) / (X_max - X_min + 1e-8)
    elif method == "zscore":
        mean = X.mean(axis = 1, keepdims = True)
        std = X.std(axis = 1, keepdims = True) + 1e-8
        
        return (X - mean) / std
    else:
        raise ValueError(f"Unknown method: {method}")

def grid_search(X_train, y_train, X_val, y_val, n_classes,  learning_rates = LEARNING_RATES, inits =["zero", "random"], 
                norms = ["none", "minmax", "zscore"]):
    results = []
    for lr in learning_rates:
        for init in inits:
            for norm in norms:
                # Apply normalisation
                X_train_norm = normalise_data(X_train, norm)
                X_val_norm = normalise_data(X_val, norm)

                n_features = X_train_norm.shape[1]
                model = MultiClassPerceptron(n_classes, n_features, learning_rate=lr)

                # Init strategy
                if init == "random":
                    for m in model.models:
                        m.weights = np.random.randn(n_features) * 0.01

                history = train_perceptron(model, X_train_norm, y_train,  X_val_norm, y_val, epochs = EPOCHS)
                best_val = max(history["val_acc"])
                results.append((lr, init, norm, best_val))
                print(f"lr={lr}, init={init}, norm={norm}, best_val={best_val:.3f}")
                
    return results

### 5. Evaluation

In [ ]:
def evaluate_model(model, X, y, label_names=None):
    """
    Evaluate a trained model on given data.
    Reports Accuracy, Precision, Recall, F1 (macro-averaged).
    Displays confusion matrix if label names are given.
    """
    y_pred = [model.predict(xi) for xi in X]

    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred, average = "macro", zero_division = 0)
    rec = recall_score(y, y_pred, average = "macro", zero_division = 0)
    f1 = f1_score(y, y_pred, average = "macro", zero_division = 0)

    print(f"Accuracy:  {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall:    {rec:.3f}")
    print(f"F1 Score:  {f1:.3f}")

    # Confusion matrix
    cm = confusion_matrix(y, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix = cm, display_labels = label_names)
    disp.plot(cmap = plt.cm.Reds, xticks_rotation = 45)
    plt.show()

    return acc, prec, rec, f1

In [ ]:
def load_subset(base_path, mode = "grayscale", max_per_class = 100):
    X, y = [], []
    train_path = os.path.join(base_path, "train")
    for label_name in os.listdir(train_path):
        label_dir = os.path.join(train_path, label_name)
        if not os.path.isdir(label_dir):
            continue
        label = label_map_inv[label_name]
        files = os.listdir(label_dir)[:max_per_class]
        for file in files:
            img_path = os.path.join(label_dir, file)
            img = Image.open(img_path)
            if mode == "grayscale":
                img = img.convert("L")
            else:
                img = img.convert("RGB")
            arr = np.array(img).flatten().astype(np.float32)
            X.append(arr)
            y.append(label)
    return np.array(X), np.array(y)

# Load small subsets (This currently loads the FULL data set) (DO we wanna test on subsets?)
X_gray, y_gray = load_dataset("data/grayscale", mode = "grayscale") # load_subset
X_rgb, y_rgb = load_dataset("data/rgb", mode = "rgb")

# Train/Test split
X_train_gray, X_test_gray, y_train_gray, y_test_gray = train_test_split(X_gray, y_gray, test_size = TEST_SIZE, random_state = RANDOM_SEED)
X_train_rgb, X_test_rgb, y_train_rgb, y_test_rgb = train_test_split(X_rgb, y_rgb, test_size = TEST_SIZE, random_state = RANDOM_SEED)

print("Grayscale subset:", X_train_gray.shape, y_train_gray.shape)
print("RGB subset:", X_train_rgb.shape, y_train_rgb.shape)

In [ ]:
# Define our models...
n_features_gray = X_train_gray.shape[1]
n_features_rgb = X_train_rgb.shape[1]
label_names = list(label_map_inv.keys())

model_gray = MultiClassPerceptron(n_classes = LABEL_COUNT, n_features = n_features_gray, learning_rate = LEARNING_RATE)
model_gray_es = MultiClassPerceptron(n_classes = LABEL_COUNT, n_features = n_features_gray, learning_rate = LEARNING_RATE)

model_rgb = MultiClassPerceptron(n_classes = LABEL_COUNT, n_features = n_features_rgb, learning_rate = LEARNING_RATE)
model_rgb_es = MultiClassPerceptron(n_classes = LABEL_COUNT, n_features = n_features_rgb, learning_rate = LEARNING_RATE)

In [ ]:
# Train grayscale epoch
print("Training grayscale perceptron using epochs...")
history_gray = train_perceptron(model_gray, X_train_gray, y_train_gray, X_test_gray, y_test_gray, epochs = EPOCHS)

print("\nValidation performance (grayscale dataset):")
evaluate_model(model_gray, X_val_gray, y_val_gray, label_names = label_names)

In [ ]:
# Train grayscale early-stopping
print("Training grayscale perceptron using early-stopping...")
history_gray_es = train_perceptron(model_gray_es, X_train_gray, y_train_gray, X_test_gray, y_test_gray, epochs = EPOCHS,
                                   stop_criteria = "early_stop", patience = 3)

print("\nValidation performance (grayscale dataset):")
evaluate_model(model_gray_es, X_val_gray, y_val_gray, label_names = label_names) # test or val here?

In [ ]:
# Train RGB epoch
print("Training RGB perceptron using epochs...")
history_rgb = train_perceptron(model_rgb, X_train_rgb, y_train_rgb, X_test_rgb, y_test_rgb, epochs = EPOCHS) # epochs = 5

print("\nValidation performance (RGB dataset):")
evaluate_model(model_rgb, X_val_rgb, y_val_rgb, label_names = label_names)

In [ ]:
# Train RGB early-stopping
print("Training RGB perceptron using early-stopping...")
history_rgb_es = train_perceptron(model_rgb_es, X_train_rgb, y_train_rgb, X_test_rgb, y_test_rgb, epochs = EPOCHS,
                                   stop_criteria = "early_stop", patience = 3)

print("\nValidation performance (RGB dataset):")
evaluate_model(model_rgb_es, X_val_rgb, y_val_rgb, label_names = label_names) # test or val here?

In [ ]:
# Compare results of fixed epochs vs early stopping
plt.plot(history_gray["val_acc"], label="Fixed epochs")
plt.plot(history_gray_es["val_acc"], label="Early stopping")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Stopping criteria comparison (grayscale)")
plt.legend()
plt.show()

plt.plot(history_rgb["val_acc"], label="Fixed epochs")
plt.plot(history_rgb_es["val_acc"], label="Early stopping")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Stopping criteria comparison (RGB)")
plt.legend()
plt.show()